# 공공데이터 활용 AI 딥러닝 기반 영상분석 기초 실습

환경 점검부터 동영상 추적까지 전체 흐름을 한 번에 진행하는 통합 실습 노트북입니다.

> 💡 시간이 부족하거나 전체 흐름을 빠르게 확인할 때 사용하세요.
> 단계별 상세 실습은 `01_` ~ `04_` 노트북을 각각 사용하세요.

> ⚠️ **실행 전 확인**
> - Jupyter는 `ITS2026/` 루트에서 실행해야 합니다.
> - `data/images/` 에 이미지, `data/videos/` 에 동영상(.mp4 권장)을 준비하세요.
> - 첫 실행 시 `yolov8n.pt` (~6MB) 가 자동 다운로드됩니다.

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd
from collections import Counter
from ultralytics import YOLO

# ── 경로 설정 ──────────────────────────────────────────────
ROOT = Path.cwd()
if not (ROOT / 'scripts').exists() and (ROOT.parent / 'scripts').exists():
    ROOT = ROOT.parent

IMAGE_DIR  = ROOT / 'data' / 'images'
VIDEO_DIR  = ROOT / 'data' / 'videos'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

image_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
video_exts = {'.mp4', '.avi', '.mov', '.wmv', '.mkv'}

images = sorted([p for p in IMAGE_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in image_exts]) \
         if IMAGE_DIR.exists() else []
videos = sorted([p for p in VIDEO_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in video_exts]) \
         if VIDEO_DIR.exists() else []

print('워크스페이스:', ROOT)
print('이미지 수:', len(images), '/ 동영상 수:', len(videos))

if not images:
    raise FileNotFoundError(f'이미지 없음 → {IMAGE_DIR} 에 파일을 넣어주세요.')
if not videos:
    print(f'⚠️  동영상 없음 → {VIDEO_DIR} 에 .mp4 파일을 넣어주세요.')
    print('   동영상 관련 셀은 건너뜁니다.')

In [ ]:
# 단일 이미지 객체검출
model = YOLO('yolov8n.pt')
image_path = images[0]
single_results = model.predict(
    source=str(image_path), conf=0.25,
    save=True, project=str(OUTPUT_DIR), name='combined_single', exist_ok=True
)
print('단일 이미지 검출 개수:', len(single_results[0].boxes))

In [ ]:
# 이미지 폴더 일괄 검출
batch_results = model.predict(
    source=str(IMAGE_DIR), conf=0.25,
    save=True, project=str(OUTPUT_DIR), name='combined_batch', exist_ok=True
)
rows    = []
counter = Counter()
for result in batch_results:
    rows.append({'image': Path(result.path).name, 'detect_count': len(result.boxes)})
    for class_id in result.boxes.cls.tolist():
        counter[result.names[int(class_id)]] += 1

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print('\n클래스별:', dict(counter))

In [ ]:
# 동영상 객체검출 (동영상이 있을 때만 실행)
if videos:
    video_path = videos[0]
    if video_path.suffix.lower() == '.wmv':
        print('⚠️  WMV 파일 — Colab에서는 재생되지 않습니다.')
    model.predict(
        source=str(video_path), conf=0.3,
        save=True, project=str(OUTPUT_DIR), name='combined_video_detect', exist_ok=True
    )
    print('동영상 검출 완료:', OUTPUT_DIR / 'combined_video_detect')
else:
    print('동영상 파일 없음 — 건너뜀')

In [ ]:
# 동영상 추적 (동영상이 있을 때만 실행)
if videos:
    model.track(
        source=str(video_path), conf=0.3,
        save=True, project=str(OUTPUT_DIR), name='combined_video_track', exist_ok=True
    )
    print('추적 완료:', OUTPUT_DIR / 'combined_video_track')
else:
    print('동영상 파일 없음 — 건너뜀')